In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'refactor-continuo'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

# 03 - Creazione e salvataggio dataset
- creazione dei record (allineamento dei dati delle colture con i dati satellitari)
- feature engineering (creazione di nuove feature)

#### Caricamento ground truth e legenda

In [ ]:
import os                                                   
import glob                                                 
import json                                                 
import re                                                   
from pathlib import Path                                    
import pandas as pd                                         
import numpy as np    

# definizione dei percorsi di input
ground_truth_path = DATA_DIR / "interim" / "points.json"
sentinel2_data_path = DATA_DIR / "processed" / "sentinel2_data"
legend_path = DATA_DIR / "processed" / "legend.json"

# caricamento del file con le coordinate e i codici delle colture
ground_truth_df = pd.read_json(ground_truth_path)

# esclusione classi fittizie 'Undecided' (3100 e 3200) e 'Other Cereals' (1150)
EXCLUDED_CLASSES = [3100, 3200, 1150]
initial_count = len(ground_truth_df)
ground_truth_df = ground_truth_df[~ground_truth_df['code'].astype(int).isin(EXCLUDED_CLASSES)]

print(f"Campi totali: {initial_count}")                     
print(f"Campi scartati (classi {EXCLUDED_CLASSES}): {initial_count - len(ground_truth_df)}")                      
print(f"Campi validi da elaborare: {len(ground_truth_df)}") 

# caricamento della legenda con i nomi delle colture
legend = {}
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)
    print(f"Legenda caricata con successo ({len(legend)} voci).")
else:
    print(f"!! Errore nel caricamento della legenda.")

#### Funzione di estrazione e feature engineering

In [ ]:
import rasterio                                             
from rasterio.warp import transform       

# regex per associare in modo univoco ciascun file TIF a mese (01..12) e banda (B02..B12)                  
BAND_FILE_PATTERN = re.compile(r"^\d{4}-(\d{2})_.*_(B02|B03|B04|B08|B11|B12)_[12]0m\.tif$")  

def process_single_point(args):
    """
    Estrae le serie temporali Sentinel-2 e calcola gli indici
    fenologici per un singolo punto.
    Garantisce l'allineamento mese-banda ed esegue l'interpolazione continua.
    """

    index, row_dict, base_data_dir, legend_map = args
    crop_id = int(row_dict["code"])

    lon_val = float(row_dict["lon"])
    lat_val = float(row_dict["lat"])

    point_dir = base_data_dir / f"point_{index}"
    if not point_dir.exists():
        return index, None

    # verifica la presenza del file unificato e dei file raw individuali
    tif_files = list(point_dir.glob("sentinel2_data_*.tif"))
    all_raw_tifs = sorted([                                 
        f for f in os.listdir(point_dir)                    
        if f.endswith(".tif") and not f.startswith("sentinel2_data_")                                 
    ])                                                      
                                                                
    monthly_bands = {m: {} for m in range(1, 13)}           
                                                                
    try:                                                    
        # --- ESTRAZIONE DEI VALORI DELLA PATCH 3x3 CENTRATA SUL CAMPO ---                                    
        if tif_files:                                       
            point_tif = tif_files[0]                        
            with rasterio.open(point_tif) as src:           
                data = src.read()  # Shape: (n_layers, height, width)                                                
                n_layers = src.count                        
                if n_layers < 6:                            
                    return index, None                      
                                                                                                   
                xs, ys = transform("EPSG:4326", src.crs, [lon_val], [lat_val])                                         
                r_center, c_center = src.index(xs[0], ys[0])
                                                            
                # controllo di sicurezza sui limiti dell'immagine                                                 
                if r_center < 0 or r_center >= data.shape[1] or c_center < 0 or c_center >= data.shape[2]:                 
                    return index, None                      
                                                            
                # finestra 3x3 pixel (30m x 30m)            
                r_min = max(0, r_center - 1)                
                r_max = min(data.shape[1], r_center + 2)    
                c_min = max(0, c_center - 1)                
                c_max = min(data.shape[2], c_center + 2)    
                                                            
                crop_patch = data[:, r_min:r_max, c_min:c_max]                                                  
                                                                
                # mediana su ciascuno strato escludendo i valori nulli (nodata)                                         
                layer_values = []                           
                for l in range(n_layers):                   
                    valid_vals = crop_patch[l][crop_patch[l] > 0]                                                          
                    layer_values.append(float(np.median(valid_vals)) if len(valid_vals) > 0 else 0.0)          
                                                            
                # mappatura rigorosa layer -> mese di calendario reale e banda                                      
                if len(all_raw_tifs) == n_layers:           
                    for l, fname in enumerate(all_raw_tifs):
                        m_match = BAND_FILE_PATTERN.match(fname)                                                  
                        if m_match and layer_values[l] > 0: 
                            mo = int(m_match.group(1))      
                            band = m_match.group(2)         
                            monthly_bands[mo][band] = layer_values[l]                                               
                elif n_layers == 72:                        
                    # 12 mesi completi x 6 bande ordinate                                              
                    band_names = ["B02", "B03", "B04", "B08", "B11", "B12"]                                                 
                    for l in range(72):                     
                        mo = (l // 6) + 1                   
                        band = band_names[l % 6]            
                        if layer_values[l] > 0:             
                            monthly_bands[mo][band] = layer_values[l]                                               
                else:                                       
                    return index, None                      
                                                    
        elif len(all_raw_tifs) >= 6:                        
            # se non è presente il raster aggregato                                                     
            for fname in all_raw_tifs:                      
                m_match = BAND_FILE_PATTERN.match(fname)    
                if not m_match:                             
                    continue                                
                mo = int(m_match.group(1))                  
                band = m_match.group(2)                     
                with rasterio.open(point_dir / fname) as src:                                                          
                    xs, ys = transform("EPSG:4326", src.crs, [lon_val], [lat_val])                                         
                    rc, cc = src.index(xs[0], ys[0])        
                    if 0 <= rc < src.height and 0 <= cc < src.width:                                                    
                        r_min, r_max = max(0, rc - 1), min(src.height, rc + 2)                                       
                        c_min, c_max = max(0, cc - 1), min(src.width, cc + 2)                                        
                        win = src.read(1, window=((r_min, r_max), (c_min, c_max)))                                      
                        valid = win[win > 0]                
                        if len(valid) > 0:                  
                            monthly_bands[mo][band] = float(np.median(valid))                                       
        else:                                               
            return index, None                              
                                                                
        # --- MEDIE ANNUALI DI RIFLETTANZA PER BANDA ---                                                     
        def extract_clean_band_mean(band_name):             
            vals = [
                monthly_bands[m][band_name] 
                for m in range(1, 13) 
                if band_name in monthly_bands[m] and monthly_bands[m][band_name] > 0
            ]                              
            return float(np.mean(vals)) if vals else 0.0    
                                                                
        record = {                                          
            "ID_Campo": index,                              
            "Ground_Truth": crop_id,                        
            "Crop_Name": legend_map.get(str(crop_id), str(crop_id)),                                                
            "Blu_B02": extract_clean_band_mean("B02"),      
            "Verde_B03": extract_clean_band_mean("B03"),    
            "Rosso_B04": extract_clean_band_mean("B04"),    
            "NIR_B08": extract_clean_band_mean("B08"),      
            "SWIR1_B11": extract_clean_band_mean("B11"),    
            "SWIR2_B12": extract_clean_band_mean("B12"),    
        }                                                   
                                                                
        # --- SERIE TEMPORALI MENSILI (NDVI e NDWI) ---                                                     
        raw_ndvis = {}                                      
        raw_ndwis = {}                                      
                                                            
        for m in range(1, 13):                              
            red_v = monthly_bands[m].get("B04", 0.0)        
            nir_v = monthly_bands[m].get("B08", 0.0)        
            swir1_v = monthly_bands[m].get("B11", 0.0)      
                                                            
            # NDVI mensile                                  
            if red_v > 0 and nir_v > 0:                     
                raw_ndvis[m] = (nir_v - red_v) / (nir_v + red_v)                                                        
            else:                                           
                raw_ndvis[m] = np.nan                       
                                                            
            # NDWI mensile (contenuto idrico fogliare)      
            if nir_v > 0 and swir1_v > 0:                   
                raw_ndwis[m] = (nir_v - swir1_v) / (nir_v + swir1_v)                                                      
            else:                                           
                raw_ndwis[m] = np.nan                       
                                                            
        s_ndvi = pd.Series(raw_ndvis, index=range(1, 13))   
        s_ndwi = pd.Series(raw_ndwis, index=range(1, 13))   
                                                                
        # vengono richiesti almeno 6 mesi con osservazioni valide                                           
        if s_ndvi.dropna().count() < 6:                     
            return index, None                              
                                                            
        # ricostruzione temporale continua tramite interpolazione lineare                                        
        s_ndvi_interp = s_ndvi.interpolate(method="linear").bfill().ffill()                                               
        s_ndwi_interp = s_ndwi.interpolate(method="linear").bfill().ffill()                                               
                                                            
        # salvataggio delle 12 feature mensili allineate al calendario (01..12)                                           
        for m in range(1, 13):                              
            month_str = f"{m:02d}"                          
            record[f"NDVI_{month_str}"] = round(float(s_ndvi_interp[m]), 4)                             
            record[f"NDWI_{month_str}"] = round(float(s_ndwi_interp[m]), 4)                             
                                                            
        ndvi_arr = s_ndvi_interp.values  # array ordinato dei 12 valori mensili                                         
        ndwi_arr = s_ndwi_interp.values                     
                                                                
        # --- INDICATORI FENOLOGICI E STRUTTURALI ---
        record["NDVI_max"] = round(float(np.max(ndvi_arr)), 4)                                                            
        record["NDVI_min"] = round(float(np.min(ndvi_arr)), 4)                                                            
        record["NDVI_amp"] = round(float(record["NDVI_max"] - record["NDVI_min"]), 4)                                       
        record["Peak_Month"] = int(np.argmax(ndvi_arr) + 1) 
        record["NDVI_mean"] = round(float(np.mean(ndvi_arr)), 4)                                                            
        record["NDVI_std"] = round(float(np.std(ndvi_arr)), 4)                                                            
                                                            
        # delta stagionali (indices: 0=Gen, 3=Apr, 4=Mag, 5=Giu, 6=Lug, 8=Set)                                          
        record["NDVI_diff_lug_apr"] = round(float(ndvi_arr[6] - ndvi_arr[3]), 4)                    
        record["NDVI_diff_apr_gen"] = round(float(ndvi_arr[3] - ndvi_arr[0]), 4)                    
        record["NDVI_diff_set_lug"] = round(float(ndvi_arr[8] - ndvi_arr[6]), 4)                    
        record["Orzo_Wheat_Ratio"] = round(float(ndvi_arr[3] / (ndvi_arr[4] + 0.01)), 4)                                   
        record["NDVI_senescence_rate"] = round(float(ndvi_arr[5] - ndvi_arr[4]), 4)                    
                                                                
        # indicatori di contenuto idrico (NDWI)             
        record["NDWI_mean"] = round(float(np.mean(ndwi_arr)), 4)                                                            
        record["NDWI_diff_lug_gen"] = round(float(ndwi_arr[6] - ndwi_arr[0]), 4)                    
                                                            
        # rapporti spettrali SWIR/NIR per lignina e biomassa
        nir_b8 = record["NIR_B08"]                          
        swir1_b11 = record["SWIR1_B11"]                     
        swir2_b12 = record["SWIR2_B12"]                     
        record["SWIR_NIR_ratio"] = round(float(swir1_b11 / (nir_b8 + 0.001) if nir_b8 > 0 else 0.0), 4)                  
        record["SWIR_Cellulose_ratio"] = round(float(swir2_b12 / (swir1_b11 + 0.001) if swir1_b11 > 0 else 0.0), 4)                                                 
                                                                
        # --- METRICHE SPECIALISTICHE PER I CLUSTER CRITICI ---                                           
        # contrasto sempreverdi (olivi) vs caducifoglie (frutteti/noci)                                               
        ndvi_winter = float((ndvi_arr[0] + ndvi_arr[1] + ndvi_arr[11]) / 3.0)                                          
        ndvi_summer = float((ndvi_arr[6] + ndvi_arr[7]) / 2.0)                                                            
        record["NDVI_winter"] = round(ndvi_winter, 4)       
        record["NDVI_summer_winter_diff"] = round(float(ndvi_summer - ndvi_winter), 4)                    
                                                            
        # discriminazione colture a ciclo breve (ortaggi)
        record["NDVI_AUC"] = round(float(np.sum(ndvi_arr)), 4)                                                            
        record["Active_Months_Count"] = int(np.sum(ndvi_arr > 0.35))                                                      
                                                                
        # differenziale di senescenza maggio/aprile (orzo vs grano)                                                     
        record["Senescence_May_Apr"] = round(float(ndvi_arr[4] - ndvi_arr[3]), 4)                    
        record["Greenup_Mar_Feb"] = round(float(ndvi_arr[2] - ndvi_arr[1]), 4)                                              
                                                            
        return index, record                                
    except Exception as e:                                  
        return index, None                                  

print("Funzione process_single_point compilata con successo.")

#### Esecuzione parallela multithread sull'intero dataset

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed                                                  
from tqdm.auto import tqdm

# preparazione della lista di task
tasks = [(index, row.to_dict(), sentinel2_data_path, legend) for index, row in ground_truth_df.iterrows()]
workers = min(16, os.cpu_count() or 4)

print(f"Avvio estrazione serie temporali su {len(tasks)} campi con {workers} worker...")

valid_results = []
with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(process_single_point, t) for t in tasks]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Elaborazione campi"):
        idx, res = future.result()
        if res is not None:
            valid_results.append(res)

# creazione del dataframe finale ordinato per ID_Campo
final_df = pd.DataFrame(valid_results)
if not final_df.empty:
    final_df = final_df.sort_values(by="ID_Campo").reset_index(drop=True)

print(f"Elaborazione completata! Dataset creato con {len(final_df)} campi validi.")

#### Salvataggio e report del dataset

In [ ]:
dataset_dir = DATA_DIR / 'processed' / 'dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

parquet_path = dataset_dir / 'dataset.parquet'
csv_path = dataset_dir / 'dataset.csv'

final_df.to_parquet(parquet_path, index=False)
final_df.to_csv(csv_path, index=False)

print(f"File salvati con successo in:")
print(f"   • Parquet: {parquet_path.resolve()}")   
print(f"   • CSV:     {csv_path.resolve()}")         
print(f"\nDimensioni tabella finale: {final_df.shape[0]} righe x {final_df.shape[1]} colonne")

# controllo campioni estratti per ciascuna coltura
print(f"Distribuzione dei campioni estratti per classe:")
display(final_df['Crop_Name'].value_counts())

# anteprima delle prime 5 righe
display(final_df.head())